# Model Training with MLflow
## Heart Disease Prediction

This notebook trains and evaluates machine learning models for heart disease prediction with MLflow experiment tracking.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    roc_auc_score, confusion_matrix, classification_report
)
import mlflow
import mlflow.sklearn
from pathlib import Path
import pickle
import sys
import os

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from src.data.preprocess import HeartDiseasePreprocessor, load_and_clean_data

print("Libraries imported successfully!")


## 1. Load and Prepare Data


In [ ]:
# Load and clean data
data_path = "../data/raw/heart_disease.csv"
print("Loading and cleaning data...")
X, y = load_and_clean_data(data_path)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


## 2. Preprocessing


In [ ]:
# Preprocess data
print("Preprocessing data...")
preprocessor = HeartDiseasePreprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Save preprocessor
output_dir = "../models"
Path(output_dir).mkdir(parents=True, exist_ok=True)
preprocessor_path = os.path.join(output_dir, "preprocessor.pkl")
preprocessor.save(preprocessor_path)
print(f"Preprocessor saved to {preprocessor_path}")
print(f"Processed train shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")


## 3. Setup MLflow


In [ ]:
# Set MLflow experiment
mlflow.set_experiment("heart_disease")
print("MLflow experiment set to: heart_disease")


## 4. Train Logistic Regression


In [ ]:
# Train Logistic Regression
with mlflow.start_run(run_name="LogisticRegression"):
    # Create model
    lr_model = LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')
    
    # Train
    lr_model.fit(X_train_processed, y_train)
    
    # Predictions
    y_train_pred = lr_model.predict(X_train_processed)
    y_test_pred = lr_model.predict(X_test_processed)
    y_test_proba = lr_model.predict_proba(X_test_processed)[:, 1]
    
    # Metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    roc_auc = roc_auc_score(y_test, y_test_proba)
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(lr_model, X_train_processed, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Log to MLflow
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("solver", "liblinear")
    mlflow.log_param("random_state", 42)
    
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("cv_roc_auc_mean", cv_mean)
    mlflow.log_metric("cv_roc_auc_std", cv_std)
    
    mlflow.sklearn.log_model(lr_model, "model")
    mlflow.log_artifact(preprocessor_path, "preprocessor")
    
    # Print results
    print("Logistic Regression Results:")
    print(f"  Train Accuracy: {train_accuracy:.4f}")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  CV ROC-AUC: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"\n  Confusion Matrix:")
    print(confusion_matrix(y_test, y_test_pred))
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_test_pred))
    
    # Save model
    lr_path = os.path.join(output_dir, "logisticregression_model.pkl")
    with open(lr_path, 'wb') as f:
        pickle.dump(lr_model, f)
    print(f"\n  Model saved to {lr_path}")
    
    lr_results = {
        'model': lr_model,
        'test_accuracy': test_accuracy,
        'roc_auc': roc_auc,
        'precision': precision,
        'recall': recall,
        'cv_mean': cv_mean
    }


## 5. Train Random Forest


In [ ]:
# Train Random Forest
with mlflow.start_run(run_name="RandomForest"):
    # Create model
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    
    # Train
    rf_model.fit(X_train_processed, y_train)
    
    # Predictions
    y_train_pred = rf_model.predict(X_train_processed)
    y_test_pred = rf_model.predict(X_test_processed)
    y_test_proba = rf_model.predict_proba(X_test_processed)[:, 1]
    
    # Metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    roc_auc = roc_auc_score(y_test, y_test_proba)
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(rf_model, X_train_processed, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Log to MLflow
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("random_state", 42)
    
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("cv_roc_auc_mean", cv_mean)
    mlflow.log_metric("cv_roc_auc_std", cv_std)
    
    mlflow.sklearn.log_model(rf_model, "model")
    mlflow.log_artifact(preprocessor_path, "preprocessor")
    
    # Print results
    print("Random Forest Results:")
    print(f"  Train Accuracy: {train_accuracy:.4f}")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  CV ROC-AUC: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"\n  Confusion Matrix:")
    print(confusion_matrix(y_test, y_test_pred))
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_test_pred))
    
    # Save model
    rf_path = os.path.join(output_dir, "randomforest_model.pkl")
    with open(rf_path, 'wb') as f:
        pickle.dump(rf_model, f)
    print(f"\n  Model saved to {rf_path}")
    
    rf_results = {
        'model': rf_model,
        'test_accuracy': test_accuracy,
        'roc_auc': roc_auc,
        'precision': precision,
        'recall': recall,
        'cv_mean': cv_mean
    }


## 6. Select Best Model


In [ ]:
# Compare models and select best
results = {
    'LogisticRegression': lr_results,
    'RandomForest': rf_results
}

best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_model = results[best_model_name]['model']

print("=" * 80)
print(f"Best Model: {best_model_name}")
print(f"  ROC-AUC: {results[best_model_name]['roc_auc']:.4f}")
print(f"  Test Accuracy: {results[best_model_name]['test_accuracy']:.4f}")
print("=" * 80)

# Save best model
best_model_path = os.path.join(output_dir, "best_model.pkl")
with open(best_model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f"\nBest model saved to {best_model_path}")


## Summary

Training complete! All models have been trained, evaluated, and logged to MLflow. The best model has been saved to `../models/best_model.pkl`.
